In [3]:
%pip install -q langgraph langchain-openai langchain-core python-dotenv grandalf

Note: you may need to restart the kernel to use updated packages.


# 02 · Manual `StateGraph`

Same agent from notebook 01 built piece by piece — without `create_react_agent`.

### New concepts
- **`TypedDict`** — defines the state shape (which keys, which types)
- **`add_messages`** — Reducer: appends instead of replacing the message list
- **`add_conditional_edges`** — routing: if there's a `tool_call` → tools node, otherwise → END

In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [4]:
from typing import Annotated, TypedDict

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

## State + Tool + LLM

In [5]:
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"Sunny, 22 °C in {city}"


class State(TypedDict):
    messages: Annotated[list, add_messages]  # Reducer: append, no replace


llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools([get_weather])

## Nodes and routing

In [6]:
def agent(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def should_continue(state: State):
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

## Build and compile the graph

In [7]:
builder = StateGraph(State)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode([get_weather]))
builder.set_entry_point("agent")
builder.add_conditional_edges("agent", should_continue)
builder.add_edge("tools", "agent")

graph = builder.compile()

## Visualize the graph (same 2 nodes as the prebuilt)

In [8]:
graph.get_graph().print_ascii()

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | agent |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


## Invoke (same input as notebook 01)

In [9]:
result = graph.invoke({"messages": [{"role": "user", "content": "Weather in Madrid?"}]})

for msg in result["messages"]:
    print(f"{msg.__class__.__name__}: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  tool_calls: {msg.tool_calls}")

HumanMessage: Weather in Madrid?
AIMessage: 
  tool_calls: [{'name': 'get_weather', 'args': {'city': 'Madrid'}, 'id': 'call_9S67sfE1kqOoZz2YMevhLVbD', 'type': 'tool_call'}]
ToolMessage: Sunny, 22 °C in Madrid
AIMessage: The current weather in Madrid is sunny with a temperature of 22 °C.


In [10]:
print(result["messages"][-1].content)

The current weather in Madrid is sunny with a temperature of 22 °C.
